# Channel-model parameter-count grid

Reads `train_and_validate.yml`, expands `CHANNEL_GRID_SEARCH.models` exactly the way
the grid search does, instantiates every point on CPU, and reports the parameter count
of each.

The number that actually matters for the pareto sweep is the octave bucket
`round(log2(num_params))`: `select_channel_models(mode="best_per_size")` keeps one
winner per `(model, prob/nonprob, octave)`, so two sizes in the same octave compete for
a single slot on the pareto plot and one of them is thrown away.

In [1]:
import math
import sys
from pathlib import Path

import yaml

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "experiments" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from modules.grid_search.adapters import MODEL_REGISTRY
from modules.grid_search.grid import expand_grid

CONFIG_PATH = REPO_ROOT / "experiments/train_and_validate.yml"
REFERENCE_FAMILY = "tcn"
SHARED_PARAMS = {"EVAL_CHUNK_SIZE": 512}  # required by adapters; irrelevant to parameter counts

full_config = yaml.safe_load(CONFIG_PATH.read_text())
model_specs = full_config["CHANNEL_GRID_SEARCH"]["models"]
print(f"{len(model_specs)} model specs in {CONFIG_PATH.name}")

11 model specs in train_and_validate.yml


In [2]:
def sweep_key(model_name, params):
    """The architecture knobs actually swept for this family, as a printable string.

    beta_nll and the training knobs are excluded: they duplicate points without
    changing the parameter count.
    """
    architecture_keys = MODEL_REGISTRY[model_name].ARCH_KEYS
    parts = [f"{key}={params[key]}" for key in architecture_keys
             if key in params and key != "distribution"]
    return " ".join(parts)


points = expand_grid(model_specs)

instantiated = []
for point in points:
    model_name = point["model"]
    params = point["params"]
    adapter = MODEL_REGISTRY[model_name].from_config(params, "cpu", shared=SHARED_PARAMS)
    num_params = int(adapter.num_params())

    instantiated.append({
        "model": model_name,
        "distribution": params.get("distribution", "none"),
        "architecture": sweep_key(model_name, params),
        "num_params": num_params,
        "octave": int(round(math.log2(num_params))),
    })

print(f"{len(instantiated)} grid points instantiated")

55 grid points instantiated


## Per-family size ladder

Distribution and `beta_nll` do not change the architecture, so the distinct sizes are
collapsed per `(family, architecture)`. `runs` is how many grid points land on that size
(nonprob + Gaussian x each beta_nll).

In [3]:
sizes_by_family = {}
for entry in instantiated:
    family = sizes_by_family.setdefault(entry["model"], {})
    row = family.setdefault(entry["architecture"], {
        "num_params": entry["num_params"],
        "octave": entry["octave"],
        "runs": 0,
    })
    row["runs"] += 1

for family_name in sorted(sizes_by_family):
    rows = sorted(sizes_by_family[family_name].items(), key=lambda item: item[1]["num_params"])
    total_runs = sum(row["runs"] for _, row in rows)
    octaves = sorted({row["octave"] for _, row in rows})

    print(f"\n{family_name.upper()}  ({len(rows)} distinct sizes, {total_runs} runs, "
          f"octaves {octaves})")
    print(f"  {'params':>8} {'octave':>7} {'runs':>5}  architecture")
    for architecture, row in rows:
        print(f"  {row['num_params']:>8} {row['octave']:>7} {row['runs']:>5}  {architecture}")


GMP  (5 distinct sizes, 5 runs, octaves [10, 11, 12, 13, 14])
    params  octave  runs  architecture
      1128      10     1  memory_linear=281 memory_nonlinear=281 nonlinearity_order=4 cross_term_depth=0
      2817      11     1  memory_linear=281 memory_nonlinear=281 nonlinearity_order=4 cross_term_depth=1
      4503      12     1  memory_linear=281 memory_nonlinear=281 nonlinearity_order=4 cross_term_depth=2
      6186      13     1  memory_linear=281 memory_nonlinear=281 nonlinearity_order=4 cross_term_depth=3
     12888      14     1  memory_linear=281 memory_nonlinear=281 nonlinearity_order=4 cross_term_depth=7

LRU  (5 distinct sizes, 10 runs, octaves [10, 11, 12, 13, 14])
    params  octave  runs  architecture
      1087      10     2  state_dim=16 hidden_dim=6 n_layers=2 dropout=0.1 r_min=0.0 r_max=1.0 max_phase=6.28
      1907      11     2  state_dim=16 hidden_dim=10 n_layers=2 dropout=0.1 r_min=0.0 r_max=1.0 max_phase=6.28
      3377      12     2  state_dim=16 hidden_dim

## Octave coverage

One column per octave. Each cell is the number of distinct sizes the family puts in that
bucket. `1` is what you want: exactly one survives `best_per_size`, so the family gets a
point on the pareto plot there. `2+` means sizes are being trained and then discarded.
Blank means the family has no point at that x position.

In [4]:
all_octaves = sorted({entry["octave"] for entry in instantiated})

counts_by_family = {}
for family_name, architectures in sizes_by_family.items():
    counts = {}
    for row in architectures.values():
        counts[row["octave"]] = counts.get(row["octave"], 0) + 1
    counts_by_family[family_name] = counts

header = "".join(f"{2 ** octave:>8}" for octave in all_octaves)
print(f"{'family':<10}{header}")
print(f"{'octave':<10}" + "".join(f"{octave:>8}" for octave in all_octaves))
print("-" * (10 + 8 * len(all_octaves)))

for family_name in sorted(counts_by_family):
    counts = counts_by_family[family_name]
    cells = "".join(f"{counts.get(octave, ''):>8}" for octave in all_octaves)
    print(f"{family_name:<10}{cells}")

reference_octaves = sorted(counts_by_family[REFERENCE_FAMILY])
print(f"\nReference ladder ({REFERENCE_FAMILY}): octaves {reference_octaves} "
      f"= {[2 ** octave for octave in reference_octaves]} params")

for family_name in sorted(counts_by_family):
    if family_name == REFERENCE_FAMILY:
        continue
    counts = counts_by_family[family_name]
    missing = [octave for octave in reference_octaves if octave not in counts]
    collided = [octave for octave, count in sorted(counts.items()) if count > 1]
    outside = [octave for octave in sorted(counts) if octave not in reference_octaves]

    print(f"\n  {family_name}:")
    print(f"    missing from reference ladder: {missing or 'none'}")
    print(f"    octaves with >1 size (wasted runs): {collided or 'none'}")
    print(f"    octaves outside the reference ladder: {outside or 'none'}")

family        1024    2048    4096    8192   16384
octave          10      11      12      13      14
--------------------------------------------------
gmp              1       1       1       1       1
lru              1       1       1       1       1
lstm             1       1       1       1       1
tcn              2       2       2       2       2
tthnet           1       1       1       1       1

Reference ladder (tcn): octaves [10, 11, 12, 13, 14] = [1024, 2048, 4096, 8192, 16384] params

  gmp:
    missing from reference ladder: none
    octaves with >1 size (wasted runs): none
    octaves outside the reference ladder: none

  lru:
    missing from reference ladder: none
    octaves with >1 size (wasted runs): none
    octaves outside the reference ladder: none

  lstm:
    missing from reference ladder: none
    octaves with >1 size (wasted runs): none
    octaves outside the reference ladder: none

  tthnet:
    missing from reference ladder: none
    octaves with >1 size 

## Search for a sweep value hitting each target octave

Scans one architecture knob per family over a candidate range and reports the value whose
parameter count sits closest to each reference octave. Use the printed values to rewrite
the family's swept list so it lands one size per octave, matching the TCN ladder.

Edit `SEARCH_SPECS` to change which knob is scanned or to widen the candidate range.

In [5]:
SEARCH_SPECS = {
    "lru": {"knob": "hidden_dim", "candidates": range(2, 81)},
    "lstm": {"knob": "hidden_dim", "candidates": range(2, 81)},
    "tthnet": {"knob": "hidden1", "candidates": range(2, 65)},
}


def base_params(family_name):
    """The first grid point of a family, used as the fixed backdrop for the scan."""
    for point in points:
        if point["model"] == family_name:
            return dict(point["params"])
    raise KeyError(f"{family_name} not present in the grid")


for family_name, spec in SEARCH_SPECS.items():
    knob = spec["knob"]
    params = base_params(family_name)
    fixed = {key: params[key] for key in MODEL_REGISTRY[family_name].ARCH_KEYS
             if key in params and key not in (knob, "distribution")}

    curve = []
    for candidate in spec["candidates"]:
        params[knob] = candidate
        adapter = MODEL_REGISTRY[family_name].from_config(params, "cpu", shared=SHARED_PARAMS)
        curve.append((candidate, int(adapter.num_params())))

    print(f"\n{family_name.upper()}  scanning {knob}, holding {fixed}")
    chosen = []
    for octave in reference_octaves:
        target = 2 ** octave
        candidate, num_params = min(curve, key=lambda item: abs(math.log2(item[1] / target)))
        chosen.append(candidate)
        landed = int(round(math.log2(num_params)))
        flag = "" if landed == octave else f"  <-- lands in octave {landed}, not {octave}"
        print(f"  octave {octave:>2} (target {target:>6}): {knob}={candidate:>3} "
              f"-> {num_params:>7} params{flag}")

    print(f"  suggested sweep:  {knob}: {chosen}")


LRU  scanning hidden_dim, holding {'state_dim': 16, 'n_layers': 2, 'dropout': 0.1, 'r_min': 0.0, 'r_max': 1.0, 'max_phase': 6.28}
  octave 10 (target   1024): hidden_dim=  6 ->    1087 params
  octave 11 (target   2048): hidden_dim= 11 ->    2132 params
  octave 12 (target   4096): hidden_dim= 19 ->    4220 params
  octave 13 (target   8192): hidden_dim= 31 ->    8312 params
  octave 14 (target  16384): hidden_dim= 49 ->   16610 params
  suggested sweep:  hidden_dim: [6, 11, 19, 31, 49]

LSTM  scanning hidden_dim, holding {'n_layers': 2, 'dropout': 0.1}
  octave 10 (target   1024): hidden_dim=  7 ->     918 params
  octave 11 (target   2048): hidden_dim= 11 ->    2146 params
  octave 12 (target   4096): hidden_dim= 15 ->    3886 params
  octave 13 (target   8192): hidden_dim= 22 ->    8163 params
  octave 14 (target  16384): hidden_dim= 31 ->   15966 params
  suggested sweep:  hidden_dim: [7, 11, 15, 22, 31]

TTHNET  scanning hidden1, holding {'window': 281, 'hidden2': 4}
  octave 10 